In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

MC_NAME = "mc_block1_micro_drag_pricing_spillovers"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")
BASE_OUT_DIR = os.getcwd()
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 20260609
rng = np.random.default_rng(SEED)

MC_CONFIG = {
    "N": 1000,
    "psi_grid_n": 200,
    "tau_int": 0.9,
    "rho_bar_range": (1.0, 5.0),
    "a_range": (0.5, 2.0),
    "c_lambda_range": (0.1, 1.0),
    "eta_lambda_range": (1.1, 2.0),
    "c_ell_range": (0.05, 0.5),
    "eta_ell_range": (1.1, 2.0),
    "psi_L_range": (0.0, 0.3),
    "Gamma_range": (0.01, 0.05),
    "k_range": (0.1, 0.5),
    "s_N_range": (0.05, 0.5),
    "kappa_c_range": (0.05, 2.0),
    "p0_range": (0.0, 1.5),
    "c_Omega_range": (0.0, 1.0),
    "eta_Omega_range": (1.1, 2.0),
    "eps_solv": 1e-12,
    "tol_mono": 1e-10,
    "tol_kkt": 1e-10,
    "tol_g": 1e-12,
    "max_bad_share": 0.05,
}

DERIV_RHO_MIN = 1e-8

def r_func(rho, a, b):
    return a * rho - b * rho**2

def r_prime(rho, a, b):
    return a - 2.0 * b * rho

def r_second(b):
    return -2.0 * b

def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)

def lam_prime(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        out[valid] = np.exp(-c_lam * v**eta_lam) * (c_lam * eta_lam * v**(eta_lam - 1.0))
    return out if arr.ndim > 0 else float(out)

def lam_second(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        f = np.exp(-c_lam * v**eta_lam)
        g = c_lam * eta_lam * v**(eta_lam - 1.0)
        g_prime = c_lam * eta_lam * (eta_lam - 1.0) * v**(eta_lam - 2.0)
        out[valid] = f * (g_prime - g**2)
    return out if arr.ndim > 0 else float(out)

def ell_func(rho, c_ell, eta_ell):
    return c_ell * rho**eta_ell

def ell_prime(rho, c_ell, eta_ell):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * v**(eta_ell - 1.0)
    return out if arr.ndim > 0 else float(out)

def ell_second(rho, c_ell, eta_ell):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * (eta_ell - 1.0) * v**(eta_ell - 2.0)
    return out if arr.ndim > 0 else float(out)

def B_func(rho, c_lam, eta_lam, c_ell, eta_ell):
    return lam_prime(rho, c_lam, eta_lam) * ell_func(rho, c_ell, eta_ell) + lam_func(rho, c_lam, eta_lam) * ell_prime(rho, c_ell, eta_ell)

def B_prime(rho, c_lam, eta_lam, c_ell, eta_ell):
    lam = lam_func(rho, c_lam, eta_lam)
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam_pp = lam_second(rho, c_lam, eta_lam)
    ell = ell_func(rho, c_ell, eta_ell)
    ell_p = ell_prime(rho, c_ell, eta_ell)
    ell_pp = ell_second(rho, c_ell, eta_ell)
    return lam_pp * ell + 2.0 * lam_p * ell_p + lam * ell_pp

def solve_rho_star(psi_val, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell, tol=1e-12, max_iter=300):
    def F(rho):
        return r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)

    F0 = r_prime(0.0, a, b) - (1.0 - psi_val) * B_func(0.0, c_lam, eta_lam, c_ell, eta_ell)
    Fhi = r_prime(rho_bar, a, b) - (1.0 - psi_val) * B_func(rho_bar, c_lam, eta_lam, c_ell, eta_ell)

    if not np.isfinite(Fhi):
        return np.nan, "nonfinite"

    if abs(F0) <= tol:
        return 0.0, "corner_low"
    if abs(Fhi) <= tol:
        return rho_bar, "corner_high"

    if F0 * Fhi < 0.0:
        lo, hi = 0.0, rho_bar
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            f_mid = F(mid)
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (hi - lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(F0):
                lo = mid
            else:
                hi = mid
        return np.nan, "no_converge"

    if F0 > 0.0 and Fhi > 0.0:
        return rho_bar, "corner_high"
    if F0 < 0.0 and Fhi < 0.0:
        return 0.0, "corner_low"

    return np.nan, "no_bracket"

def check_kkt_status(psi_val, rho, status, a, b, c_lam, eta_lam, c_ell, eta_ell, tol_kkt=1e-10):
    F_val = r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)
    if status == "interior":
        return abs(F_val) <= tol_kkt
    elif status == "corner_low":
        return F_val <= tol_kkt
    elif status == "corner_high":
        return F_val >= -tol_kkt
    return False

def draw_primitives(rng_local, cfg):
    return {
        "rho_bar": rng_local.uniform(*cfg["rho_bar_range"]),
        "a": rng_local.uniform(*cfg["a_range"]),
        "c_lambda": rng_local.uniform(*cfg["c_lambda_range"]),
        "eta_lambda": rng_local.uniform(*cfg["eta_lambda_range"]),
        "c_ell": rng_local.uniform(*cfg["c_ell_range"]),
        "eta_ell": rng_local.uniform(*cfg["eta_ell_range"]),
        "psi_L": rng_local.uniform(*cfg["psi_L_range"]),
        "Gamma": rng_local.uniform(*cfg["Gamma_range"]),
        "k": rng_local.uniform(*cfg["k_range"]),
        "s_N": rng_local.uniform(*cfg["s_N_range"]),
        "kappa_c": rng_local.uniform(*cfg["kappa_c_range"]),
        "p0": rng_local.uniform(*cfg["p0_range"]),
        "c_Omega": rng_local.uniform(*cfg["c_Omega_range"]),
        "eta_Omega": rng_local.uniform(*cfg["eta_Omega_range"]),
    }

def draw_admissible_primitives(rng_local, cfg):
    """
    Ensures draw satisfies strict analytical admissibility from Appendix B:
    ell(rho_bar) < 1 AND k * ell(rho_bar) < 1.
    """
    while True:
        theta = draw_primitives(rng_local, cfg)
        rho_bar = theta["rho_bar"]
        c_ell = theta["c_ell"]
        eta_ell = theta["eta_ell"]
        k = theta["k"]

        ell_bar = ell_func(rho_bar, c_ell, eta_ell)
        if np.isfinite(ell_bar) and (0.0 < ell_bar < 1.0) and (k * ell_bar < 1.0):
            theta["b"] = theta["a"] / (2.0 * rho_bar)
            return theta

def evaluate_scenario(theta, cfg, with_pricing=False, with_spillovers=False):
    rho_bar = theta["rho_bar"]
    a, b = theta["a"], theta["b"]
    c_lam, eta_lam = theta["c_lambda"], theta["eta_lambda"]
    c_ell, eta_ell = theta["c_ell"], theta["eta_ell"]
    psi_L, Gamma = theta["psi_L"], theta["Gamma"]
    k, s_N = theta["k"], theta["s_N"]
    kappa_c = theta["kappa_c"]
    p0 = theta["p0"] if with_pricing else 0.0

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])
    dpsi = psi_grid[1] - psi_grid[0]

    rho_star_path = np.zeros_like(psi_grid)
    solver_status = []
    soc_ok = True
    kkt_ok = True
    deriv_invalid_any = False

    counts = {
        "nonfinite": 0, "no_bracket": 0, "no_converge": 0,
        "corner_low": 0, "corner_high": 0, "interior": 0
    }

    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell, tol=cfg["tol_kkt"])
    if st0 not in ["interior", "corner_low", "corner_high"]:
        return {"baseline_failed": True, "baseline_status": st0}

    base_drag_const = k * lam_func(rho0, c_lam, eta_lam) * ell_func(rho0, c_ell, eta_ell)

    psi_eff_path = (1.0 - p0) * psi_grid if with_pricing else psi_grid

    for i, psi in enumerate(psi_grid):
        psi_eff = psi_eff_path[i]

        rho_i, status_i = solve_rho_star(psi_eff, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell, tol=cfg["tol_kkt"])
        rho_star_path[i] = rho_i
        solver_status.append(status_i)
        counts[status_i] = counts.get(status_i, 0) + 1

        if status_i not in ["interior", "corner_low", "corner_high"]:
            deriv_invalid_any = True
            continue

        if not check_kkt_status(psi_eff, rho_i, status_i, a, b, c_lam, eta_lam, c_ell, eta_ell, tol_kkt=cfg["tol_kkt"]):
            kkt_ok = False

        if status_i == "interior":
            if rho_i >= DERIV_RHO_MIN:
                Bp = B_prime(rho_i, c_lam, eta_lam, c_ell, eta_ell)
                soc_val = r_second(b) - (1.0 - psi_eff) * Bp
                if not (np.isfinite(soc_val) and soc_val < 0.0):
                    soc_ok = False
            else:
                deriv_invalid_any = True

    solver_status = np.array(solver_status)
    ok_share = np.mean(np.isin(solver_status, ["interior", "corner_low", "corner_high"]))
    if ok_share < (1.0 - cfg["max_bad_share"]) or deriv_invalid_any:
         return {"grid_solver_failed": True, "counts": counts}

    lam_path = lam_func(rho_star_path, c_lam, eta_lam)
    ell_path = ell_func(rho_star_path, c_ell, eta_ell)

    rK_path = r_func(rho_star_path, a, b) - (1.0 - psi_eff_path) * lam_path * ell_path

    drag_path = k * lam_path * ell_path - base_drag_const

    d_rho = np.gradient(rho_star_path, dpsi)
    d_drag = np.gradient(drag_path, dpsi)
    d_rK = np.gradient(rK_path, dpsi)

    if with_pricing and np.abs(1.0 - p0) < 1e-10:
        mono_rho = np.all(np.abs(d_rho) <= cfg["tol_mono"])
        mono_drag = np.all(np.abs(d_drag) <= cfg["tol_mono"])
        mono_rK = np.all(np.abs(d_rK) <= cfg["tol_mono"])
    else:
        sign_factor = -1.0 if (with_pricing and (1.0 - p0) < 0) else 1.0
        mono_rho = np.all(sign_factor * d_rho >= -cfg["tol_mono"])
        mono_drag = np.all(sign_factor * d_drag >= -cfg["tol_mono"])
        mono_rK = np.all(sign_factor * d_rK >= -cfg["tol_mono"])

    solvency_margins = s_N - psi_grid * k * ell_path
    full_interval_stress_test_ok = np.all(solvency_margins > cfg["eps_solv"])
    fiscal_feas_share = np.mean(solvency_margins > cfg["eps_solv"])

    f_int = np.mean(solver_status == "interior")
    interior_ok = f_int >= cfg["tau_int"]

    micro_full_domain_ok = bool(
        soc_ok and kkt_ok and mono_rho and mono_drag and mono_rK and interior_ok
    )

    psi_plus_mask = (Gamma - drag_path) > 0.0
    feas_mask = solvency_margins > cfg["eps_solv"]

    psi_plus_share = np.mean(psi_plus_mask)
    feas_share = np.mean(feas_mask)

    corridor_mask = np.zeros_like(psi_grid, dtype=bool)
    for i in range(len(psi_grid)):
        if psi_plus_mask[i] and feas_mask[i]:
            corridor_mask[i] = True
        else:
            break

    mp_corridor_share = np.mean(corridor_mask)
    corridor_exists = np.any(corridor_mask)

    is_over_internalized = with_pricing and (p0 > 1.0)

    if is_over_internalized:
        g_tilde_path = np.full_like(psi_grid, np.nan)
        W_tilde_path = np.full_like(psi_grid, np.nan)
        W_tilde_monotonic = np.nan
        endpoints_straddle = np.nan
        unique_crossing = np.nan
        psi_bar_hat = np.nan
        extended_wedge_ok = np.nan
        closure_applicable = False
    else:
        closure_applicable = True
        g_tilde_path = np.zeros_like(psi_grid)
        for i, psi in enumerate(psi_grid):
            Gamma_eff_i = Gamma - drag_path[i]
            psi_closure = psi_eff_path[i] if with_pricing else psi

            if Gamma_eff_i > 0.0:
                if psi_closure == 0.0:
                    g_tilde_path[i] = Gamma_eff_i
                else:
                    disc = 1.0 + 4.0 * kappa_c * psi_closure * Gamma_eff_i
                    if disc >= 0.0:
                        g_tilde_path[i] = (-1.0 + np.sqrt(disc)) / (2.0 * kappa_c * psi_closure)
                    else:
                        g_tilde_path[i] = 0.0
            else:
                g_tilde_path[i] = 0.0

        W_tilde_path = rK_path - g_tilde_path
        dW_tilde = np.gradient(W_tilde_path, dpsi)

        W_tilde_monotonic = bool(np.all(dW_tilde >= -cfg["tol_mono"]))

        W_signs = np.sign(np.where(np.abs(W_tilde_path) <= 1e-10, 0.0, W_tilde_path))
        sign_changes = np.where(W_signs[:-1] * W_signs[1:] < 0)[0]
        endpoints_straddle = bool((W_signs[0] * W_signs[-1] < 0) or (W_signs[0] == 0) or (W_signs[-1] == 0))
        unique_crossing = bool(len(sign_changes) == 1)

        psi_bar_hat = np.nan
        if unique_crossing:
            idx = sign_changes[0]
            p_lo, p_hi = psi_grid[idx], psi_grid[idx+1]
            W_lo, W_hi = W_tilde_path[idx], W_tilde_path[idx+1]
            psi_bar_hat = p_lo - W_lo * (p_hi - p_lo) / (W_hi - W_lo)

        extended_wedge_ok = bool(W_tilde_monotonic and (unique_crossing or (not endpoints_straddle)))

    spillover_share = 0.0
    spill_midpoint_val = False
    if with_spillovers:
        c_Omega = theta["c_Omega"]
        eta_Omega = theta["eta_Omega"]
        Omega_p = c_Omega * eta_Omega * rho_star_path**(eta_Omega - 1.0)
        B_vals = B_func(rho_star_path, c_lam, eta_lam, c_ell, eta_ell)

        spill_dom = Omega_p > psi_eff_path * B_vals
        spillover_share = np.mean(spill_dom)
        spill_midpoint_val = bool(spill_dom[len(psi_grid)//2])

    if with_pricing:
        unpriced_return_at_eff = r_func(rho_star_path, a, b) - (1.0 - psi_eff_path) * lam_path * ell_path
        assert np.allclose(rK_path, unpriced_return_at_eff, atol=1e-12), "Pricing Isomorphism Identity Violation."

    return {
        "baseline_failed": False,
        "baseline_status": st0,
        "grid_solver_failed": False,
        "rho_star_path": rho_star_path,
        "drag_path": drag_path,
        "rK_path": rK_path,
        "g_tilde_path": g_tilde_path,
        "W_tilde_path": W_tilde_path,
        "micro_full_domain_ok": micro_full_domain_ok,
        "mono_rho": mono_rho,
        "mono_drag": mono_drag,
        "mono_rK": mono_rK,
        "f_int": f_int,
        "interior_ok": interior_ok,
        "soc_ok": soc_ok,
        "kkt_ok": kkt_ok,
        "full_interval_stress_test_ok": full_interval_stress_test_ok,
        "fiscal_feas_share": fiscal_feas_share,
        "psi_plus_share": psi_plus_share,
        "feas_share": feas_share,
        "mp_corridor_share": mp_corridor_share,
        "corridor_exists": corridor_exists,
        "corridor_mask": corridor_mask,
        "W_tilde_monotonic": W_tilde_monotonic,
        "endpoints_straddle": endpoints_straddle,
        "unique_crossing": unique_crossing,
        "psi_bar_hat": psi_bar_hat,
        "extended_wedge_ok": extended_wedge_ok,
        "closure_applicable": closure_applicable,
        "spillover_share": spillover_share,
        "spill_midpoint_val": spill_midpoint_val,
        "underpricing_regime": p0 < 1.0,
        "counts": counts,
    }

def run_monte_carlo(thetas, cfg, with_pricing=False, with_spillovers=False, variant_label="variant", progress_every=100):
    results = []
    baseline_fails = 0
    grid_fails = 0

    audit_summary = {
        "nonfinite": 0, "no_bracket": 0, "no_converge": 0,
        "corner_low": 0, "corner_high": 0, "interior": 0
    }

    for idx, theta in enumerate(thetas):

        if progress_every and ((idx + 1) % progress_every == 0 or idx == 0 or idx + 1 == len(thetas)):
            print(f"[{variant_label}] draw {idx + 1}/{len(thetas)}")

        res = evaluate_scenario(theta, cfg, with_pricing=with_pricing, with_spillovers=with_spillovers)

        if res.get("baseline_failed", False):
            baseline_fails += 1
            continue
        if res.get("grid_solver_failed", False):
            grid_fails += 1
            for k in audit_summary.keys():
                audit_summary[k] += res["counts"].get(k, 0)
            continue

        for k in audit_summary.keys():
            audit_summary[k] += res["counts"].get(k, 0)

        flat_res = {**theta}
        flat_res["idx"] = idx
        flat_res["micro_full_domain_ok"] = res["micro_full_domain_ok"]
        flat_res["mono_rho"] = res["mono_rho"]
        flat_res["mono_drag"] = res["mono_drag"]
        flat_res["mono_rK"] = res["mono_rK"]
        flat_res["f_int"] = res["f_int"]
        flat_res["interior_ok"] = res["interior_ok"]
        flat_res["soc_ok"] = res["soc_ok"]
        flat_res["kkt_ok"] = res["kkt_ok"]
        flat_res["full_interval_stress_test_ok"] = res["full_interval_stress_test_ok"]
        flat_res["fiscal_feas_share"] = res["fiscal_feas_share"]
        flat_res["psi_plus_share"] = res["psi_plus_share"]
        flat_res["feas_share"] = res["feas_share"]
        flat_res["mp_corridor_share"] = res["mp_corridor_share"]
        flat_res["corridor_exists"] = res["corridor_exists"]
        flat_res["W_tilde_monotonic"] = res["W_tilde_monotonic"]
        flat_res["endpoints_straddle"] = res["endpoints_straddle"]
        flat_res["unique_crossing"] = res["unique_crossing"]
        flat_res["psi_bar_hat"] = res["psi_bar_hat"]
        flat_res["extended_wedge_ok"] = res["extended_wedge_ok"]
        flat_res["closure_applicable"] = res["closure_applicable"]
        flat_res["spillover_share"] = res["spillover_share"]
        flat_res["spill_midpoint_val"] = res["spill_midpoint_val"]
        flat_res["underpricing_regime"] = res["underpricing_regime"]
        flat_res["rK_mid"] = res["rK_path"][cfg["psi_grid_n"] // 2]
        flat_res["g_tilde_mid"] = res["g_tilde_path"][cfg["psi_grid_n"] // 2]
        flat_res["drag_max"] = np.max(res["drag_path"])

        results.append(flat_res)

    df = pd.DataFrame(results)
    audit = {
        "total_drawn": len(thetas),
        "baseline_fails": baseline_fails,
        "grid_fails": grid_fails,
        "included_economies": len(df),
        **{"audit_" + k: v for k, v in audit_summary.items()}
    }
    return df, audit

print("Generating shared admissible scenarios...")
shared_thetas = [draw_admissible_primitives(rng, MC_CONFIG) for _ in range(MC_CONFIG["N"])]

print("Running baseline_unpriced...")
df_base, audit_base = run_monte_carlo(
    shared_thetas, MC_CONFIG,
    with_pricing=False,
    with_spillovers=False,
    variant_label="baseline_unpriced"
)

print("Running pricing_reoptimized_all...")
df_prem, audit_prem = run_monte_carlo(
    shared_thetas, MC_CONFIG,
    with_pricing=True,
    with_spillovers=False,
    variant_label="pricing_reoptimized_all"
)

print("Running spillovers_active...")
df_spill, audit_spill = run_monte_carlo(
    shared_thetas, MC_CONFIG,
    with_pricing=False,
    with_spillovers=True,
    variant_label="spillovers_active"
)

def bool_rate(s):
    if s is None or len(s) == 0:
        return 0.0
    s_clean = s.dropna()
    if len(s_clean) == 0:
        return np.nan
    return float(s_clean.astype(bool).astype(float).mean())

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

def generate_compilation_summary(df, label):
    N = len(df)
    if N == 0:
        return {
            "variant": label, "included_N": 0,
            "micro_full_domain_success_rate": 0.0, "micro_ci_lower": 0.0, "micro_ci_upper": 0.0,
            "full_interval_stress_test_pass_rate": 0.0, "stress_ci_lower": 0.0, "stress_ci_upper": 0.0,
            "prefix_corridor_existence_rate": 0.0, "mean_prefix_corridor_span": 0.0,
            "extended_wedge_monotonicity_rate": 0.0, "extended_wedge_diagnostic_success_rate": 0.0,
            "unique_crossing_rate_extended_wedge": 0.0,
        }

    micro_rate = bool_rate(df["micro_full_domain_ok"])
    stress_rate = bool_rate(df["full_interval_stress_test_ok"])
    corr_rate = bool_rate(df["corridor_exists"])

    mono_rate = bool_rate(df["W_tilde_monotonic"])
    wedge_rate = bool_rate(df["extended_wedge_ok"])
    crossing_rate = bool_rate(df["unique_crossing"])

    micro_ok_count = int(np.round(micro_rate * N)) if np.isfinite(micro_rate) else 0
    stress_ok_count = int(np.round(stress_rate * N)) if np.isfinite(stress_rate) else 0
    ci_micro = wilson_ci(micro_ok_count, N)
    ci_stress = wilson_ci(stress_ok_count, N)

    return {
        "variant": label,
        "included_N": N,
        "micro_full_domain_success_rate": micro_rate,
        "micro_ci_lower": ci_micro[0],
        "micro_ci_upper": ci_micro[1],
        "full_interval_stress_test_pass_rate": stress_rate,
        "stress_ci_lower": ci_stress[0],
        "stress_ci_upper": ci_stress[1],
        "prefix_corridor_existence_rate": corr_rate,
        "mean_prefix_corridor_span": df["mp_corridor_share"].mean(),
        "extended_wedge_monotonicity_rate": mono_rate,
        "extended_wedge_diagnostic_success_rate": wedge_rate,
        "unique_crossing_rate_extended_wedge": crossing_rate,
    }

df_prem_under = df_prem[df_prem["p0"] < 1.0] if len(df_prem) > 0 else pd.DataFrame()
df_prem_over = df_prem[df_prem["p0"] > 1.0] if len(df_prem) > 0 else pd.DataFrame()

print("Saving outputs...")
summary_rows = [
    generate_compilation_summary(df_base, "baseline_unpriced"),
    generate_compilation_summary(df_prem_under, "pricing_underpriced_p0_lt_1"),
    generate_compilation_summary(df_prem_over, "pricing_overinternalized_p0_gt_1"),
    generate_compilation_summary(df_spill, "spillovers_active")
]
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(OUT_DIR, "compilation_summary.csv"), index=False)

df_audit = pd.DataFrame([
    {"variant": "baseline_unpriced", **audit_base},
    {"variant": "pricing_reoptimized_all", **audit_prem},
    {"variant": "spillovers_active", **audit_spill},
])
df_audit.to_csv(os.path.join(OUT_DIR, "solver_audit_ledger.csv"), index=False)

pricing_cols = [
    "idx",
    "micro_full_domain_ok",
    "full_interval_stress_test_ok",
    "underpricing_regime",
    "p0",
]

micro_cols = ["idx", "micro_full_domain_ok", "mono_rho", "mono_drag", "mono_rK", "f_int", "interior_ok", "soc_ok", "kkt_ok"]
df_base[micro_cols].to_csv(os.path.join(OUT_DIR, "domain_summary.csv"), index=False)

stress_cols = ["idx", "full_interval_stress_test_ok", "fiscal_feas_share", "k", "s_N"]
df_base[stress_cols].to_csv(os.path.join(OUT_DIR, "stress_test_summary.csv"), index=False)

corridor_cols = ["idx", "corridor_exists", "mp_corridor_share", "psi_plus_share", "feas_share"]
df_base[corridor_cols].to_csv(os.path.join(OUT_DIR, "positive_root_corridor_summary.csv"), index=False)

wedge_cols = ["idx", "W_tilde_monotonic", "endpoints_straddle", "unique_crossing", "psi_bar_hat", "extended_wedge_ok", "closure_applicable"]
df_base[wedge_cols].to_csv(os.path.join(OUT_DIR, "extended_wedge_full_domain_summary.csv"), index=False)

df_prem_under[pricing_cols].to_csv(os.path.join(OUT_DIR, "pricing_reoptimization_summary_underpriced.csv"), index=False) if len(df_prem_under) > 0 else None
df_prem_over[pricing_cols].to_csv(os.path.join(OUT_DIR, "pricing_reoptimization_summary_overinternalized.csv"), index=False) if len(df_prem_over) > 0 else None

spill_cols = ["idx", "spillover_share", "spill_midpoint_val", "c_Omega", "eta_Omega"]
df_spill[spill_cols].to_csv(os.path.join(OUT_DIR, "spillover_summary.csv"), index=False)

df_base_merged = pd.DataFrame(shared_thetas)
df_base_merged["idx"] = df_base_merged.index
df_base_merged = df_base_merged.merge(
    df_base[["idx", "full_interval_stress_test_ok", "micro_full_domain_ok"]],
    on="idx", how="left"
).fillna({"full_interval_stress_test_ok": False, "micro_full_domain_ok": False})

stress_passed = df_base_merged[df_base_merged["full_interval_stress_test_ok"]]
stress_failed = df_base_merged[~df_base_merged["full_interval_stress_test_ok"]]

comparison_cols = ["rho_bar", "a", "c_lambda", "c_ell", "psi_L", "Gamma", "k", "s_N"]
z_scores = {}
for col in comparison_cols:
    mu_p = stress_passed[col].mean() if len(stress_passed) > 0 else 0.0
    sigma_p = stress_passed[col].std() if len(stress_passed) > 1 else 1.0
    mu_f = stress_failed[col].mean() if len(stress_failed) > 0 else 0.0
    if np.isfinite(sigma_p) and sigma_p > 0:
        z_scores[col] = (mu_f - mu_p) / sigma_p
    else:
         z_scores[col] = np.nan

df_z = pd.DataFrame(list(z_scores.items()), columns=["parameter", "z_score_failed_vs_passed"])
df_z.to_csv(os.path.join(OUT_DIR, "standardized_distances_stress_test.csv"), index=False)

crosstab_counts = pd.crosstab(df_base_merged["micro_full_domain_ok"], df_base_merged["full_interval_stress_test_ok"])
crosstab_props = pd.crosstab(df_base_merged["micro_full_domain_ok"], df_base_merged["full_interval_stress_test_ok"], normalize="all")
crosstab_counts.to_csv(os.path.join(OUT_DIR, "overlap_matrix_counts.csv"))
crosstab_props.to_csv(os.path.join(OUT_DIR, "overlap_matrix_props.csv"))

df_plot_source = df_prem_under if not df_prem_under.empty else df_base

if not df_plot_source.empty:
    plt.style.use('grayscale')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

    crossing_draws = df_plot_source[df_plot_source["full_interval_stress_test_ok"] & df_plot_source["unique_crossing"]]
    if not crossing_draws.empty:
        sample_idx = int(crossing_draws["idx"].iloc[0])
    else:
        ok_draws = df_plot_source[df_plot_source["full_interval_stress_test_ok"]]
        if not ok_draws.empty:
            sample_idx = int(ok_draws["idx"].iloc[0])
        else:
            sample_idx = int(df_plot_source["idx"].iloc[0])

    sample_theta = shared_thetas[sample_idx]
    res_unpriced = evaluate_scenario(sample_theta, MC_CONFIG, with_pricing=False)
    res_priced = evaluate_scenario(sample_theta, MC_CONFIG, with_pricing=True)
    psi_grid = np.linspace(sample_theta["psi_L"], 1.0, MC_CONFIG["psi_grid_n"])

    ax1.plot(psi_grid, res_unpriced["W_tilde_path"], color="black", linestyle="-", label=r"Unpriced Baseline ($\widetilde{W}^*$)")
    ax1.plot(psi_grid, res_priced["W_tilde_path"], color="gray", linestyle="--", label=r"Re-optimized Pricing ($\widetilde{W}^*_{priced}$)")
    ax1.axhline(0, color="lightgray", linestyle=":", linewidth=1)
    if np.isfinite(res_unpriced["psi_bar_hat"]):
        ax1.axvline(res_unpriced["psi_bar_hat"], color="black", linestyle="-.", label=r"Baseline Crossing ($\overline{\psi}$)")
    ax1.set_xlabel(r"Statutory Asymmetry ($\psi$)")
    ax1.set_ylabel("Extended Returns-Growth Wedge")
    ax1.set_title("Monotone Extended Wedge Paths")
    ax1.legend(frameon=True, facecolor="white", edgecolor="none")

    ax2.plot(psi_grid, res_unpriced["g_tilde_path"], color="black", linestyle="-", label=r"Extended Ceiling ($\widetilde{g}^*$)")
    if res_unpriced["corridor_exists"]:
        corridor_mask = res_unpriced["corridor_mask"]
        ax2.fill_between(psi_grid, 0, res_unpriced["g_tilde_path"], where=corridor_mask, color="gray", alpha=0.15, label="Positive-Root Prefix Corridor")

    ax2.set_xlabel(r"Statutory Asymmetry ($\psi$)")
    ax2.set_ylabel("Growth Rate")
    ax2.set_title("System Growth Dynamics")
    ax2.legend(frameon=True, facecolor="white", edgecolor="none")

    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "mc_block1_micro_drag_pricing_spillovers_dynamics.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_DIR, "mc_block1_micro_drag_pricing_spillovers_dynamics.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

print("\n" + "=" * 80)
print("REBUILT BLOCK 1 MONTE CARLO ENGINE EXECUTION COMPLETE")
print(f"Timestamp: {RUN_TS} | Output Directory: {OUT_DIR}")
print("-" * 80)
print(df_summary.to_string(index=False))
print("-" * 80)
print("SOLVER AUDIT LEDGER:")
print(df_audit.to_string(index=False))
print("=" * 80)
